<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_1/lessons/lesson_06_dicts_loops_comprehensions/note_lesson_06_dicts_loops_comprehensions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Урок 6 — Словники, for, comprehensions

> За цей урок: після списків/кортежів/множин (Урок 5) додаємо останній базовий контейнер — `dict`, п'ять патернів мислення для обробки колекцій (Iteration/Mapping/Aggregation/Counting/Grouping), і компактнішу форму запису циклу — comprehensions. Усе це одразу застосовуємо, будуючи гру «Minesweeper» крок за кроком — той самий скрипт, який Урок 7 розкладе на функції.

Структура уроку: **RETRIEVE → CONCEPT → CREATE → TRANSFER** (та сама послідовність, що й в Уроках 3, 4, 5, 7).

## 🔁 RETRIEVE — пригадай Урок 5 (без підглядання)

1. Чим `tuple` принципово відрізняється від `list` — не за синтаксисом, а за призначенням?
2. Що поверне `{1, 2, 2, 3}` — скільки елементів у результаті?
3. Що робить `while len(bombs) < 10: bombs.add(...)` — чому саме `while`, а не `for`?

<details>
<summary>Відповіді</summary>

1. <code>tuple</code> незмінний — використовується для запису, що вже зафіксований (факт, що стався) і не повинен випадково змінитись; <code>list</code> — для послідовності, яка змінюється по ходу програми.
2. <code>3</code> елементи — <code>set</code> автоматично видаляє дублікати, лишається <code>{1, 2, 3}</code>.
3. Ми не знаємо наперед, скільки ітерацій знадобиться (випадкові числа можуть повторюватись) — потрібно повторювати, **поки не виконається умова** (набереться 10 унікальних позицій), а не заданий наперед раз — це і є ознака <code>while</code>, а не <code>for</code>.

</details>

## 📖 CONCEPT

### 1. `dict` — четвертий контейнер: доступ за ключем, а не за позицією

У списку елемент шукають за **позицією** (`fruits[0]`). У словнику — за **ключем**, будь-яким незмінним значенням (найчастіше рядком):

In [1]:
student = {"name": "Олена", "age": 20, "group": "КН-21"}

print(student["name"])   # доступ за ключем
print(student)

# Звернення до відсутнього ключа — помилка, так само як IndexError у списку за межами
try:
    print(student["email"])
except KeyError as e:
    print(f"KeyError: {e}")

Олена
{'name': 'Олена', 'age': 20, 'group': 'КН-21'}
KeyError: 'email'


### 2. `.get()` — безпечний доступ

`d[key]` кидає `KeyError`, якщо ключа немає. `d.get(key, default)` замість помилки повертає запасне значення:

In [2]:
print(student.get("email"))            # ключа немає -> None (default за замовчуванням)
print(student.get("email", "немає"))   # ключа немає -> "немає"
print(student.get("name", "немає"))    # ключ є -> звичайне значення, default ігнорується

None
немає
Олена


### 3. Ітерація по словнику — три способи, різний результат

```
for key in d:          # тільки ключі
for val in d.values():  # тільки значення
for k, v in d.items():  # пари (ключ, значення)
```

**Типова пастка:** якщо написати `for k, v in d` (без `.items()`), Python спробує розпакувати сам **ключ** (рядок) у дві змінні `k, v` — і це впаде з `ValueError`, якщо ключ не рівно 2 символи. Подивимось на це напряму:

In [3]:
scores = {"Alice": 95, "Bob": 82, "Cara": 77}

print("for key in d:")
for key in scores:
    print(" ", key)

print()
print("for val in d.values():")
for val in scores.values():
    print(" ", val)

print()
print("for k, v in d.items():")
for k, v in scores.items():
    print(" ", k, "->", v)

print()
print("Пастка — for k, v in d (без .items()):")
try:
    for k, v in scores:
        print(k, v)
except ValueError as e:
    print(f"  ValueError: {e}")
    print('  Причина: for key in d дає рядки типу "Alice" (5 символів) — розпакувати їх у 2 змінні не можна.')

for key in d:
  Alice
  Bob
  Cara

for val in d.values():
  95
  82
  77

for k, v in d.items():
  Alice -> 95
  Bob -> 82
  Cara -> 77

Пастка — for k, v in d (без .items()):
  ValueError: too many values to unpack (expected 2)
  Причина: for key in d дає рядки типу "Alice" (5 символів) — розпакувати їх у 2 змінні не можна.


### 4. `.setdefault()` — читання з автоматичним створенням

`d.setdefault(key, default)`: якщо ключ **є** — просто повертає його значення (нічого не змінює). Якщо ключа **нема** — створює `d[key] = default` і одразу повертає це `default`. Найкорисніше, коли значення — список, який ми одразу наповнюємо:

In [4]:
teams = {}

teams.setdefault("Backend", []).append("Олег")
teams.setdefault("Backend", []).append("Ірина")   # ключ вже є -> список не перестворюється
teams.setdefault("Frontend", []).append("Марко")

print(teams)
# {'Backend': ['Олег', 'Ірина'], 'Frontend': ['Марко']}

{'Backend': ['Олег', 'Ірина'], 'Frontend': ['Марко']}


### 5. Патерн Counting — підрахунок частоти

Ідея: перебираємо елементи, для кожного збільшуємо лічильник у словнику `{значення: скільки_разів}`. `dict.get(key, 0) + 1` — стандартний прийом "прочитати поточне значення, або 0, якщо ще не було":

In [5]:
text = "banana"
counts = {}

for char in text:
    counts[char] = counts.get(char, 0) + 1   # був 0 -> стає 1, був N -> стає N+1

print(counts)
# {'b': 1, 'a': 3, 'n': 2}

# Найчастіший символ — max() з ключем "за яким значенням шукати максимум"
most_common = max(counts, key=counts.get)
print(f"Найчастіший символ: {most_common!r} ({counts[most_common]} разів)")

{'b': 1, 'a': 3, 'n': 2}
Найчастіший символ: 'a' (3 разів)


### 6. Патерн Grouping — групування записів за ключем

Розширення Counting: замість того щоб просто рахувати, збираємо самі значення у списки за ключем — `.setdefault(key, []).append(item)`:

In [6]:
numbers = [4, 15, 22, 7, 30, 3, 18, 9]

by_parity = {}
for n in numbers:
    key = "парні" if n % 2 == 0 else "непарні"
    by_parity.setdefault(key, []).append(n)

print(by_parity)
# {'парні': [4, 22, 30, 18], 'непарні': [15, 7, 3, 9]}

{'парні': [4, 22, 30, 18], 'непарні': [15, 7, 3, 9]}


### 7. `for` проти `while` — та сама задача, дві мови

`for` керується **колекцією** (Python сам бере наступний елемент). `while` керується **умовою** (ви самі відповідаєте, коли зупинитись). Та сама задача (пройти по трьох елементах) обома способами:

In [7]:
letters = ["c", "a", "t"]

print("for (Python сам керує):")
for letter in letters:
    print(" ", letter)

print()
print("while (керуємо самі, легко забути i += 1 -> нескінченний цикл):")
i = 0
while i < len(letters):
    print(" ", letters[i])
    i += 1

for (Python сам керує):
  c
  a
  t

while (керуємо самі, легко забути i += 1 -> нескінченний цикл):
  c
  a
  t


### 8. List comprehension — той самий цикл, компактніший запис

Спочатку задача звичайним циклом: **квадрати парних чисел від 1 до 10**:

In [8]:
even_squares = []
for x in range(1, 11):
    if x % 2 == 0:
        even_squares.append(x ** 2)

print(even_squares)
# [4, 16, 36, 64, 100]

[4, 16, 36, 64, 100]


Той самий результат, одним рядком — **та сама ідея**, лише компактніше записана:

```
[ що_зберегти   for елемент in колекція   if умова ]
```

In [9]:
even_squares_v2 = [x ** 2 for x in range(1, 11) if x % 2 == 0]

print(even_squares_v2)
assert even_squares_v2 == even_squares
print("Той самий результат, що й циклом вище")

[4, 16, 36, 64, 100]
Той самий результат, що й циклом вище


Comprehension може бути й без фільтра (тільки трансформація), і без трансформації (тільки фільтр):

In [10]:
squares = [x ** 2 for x in range(1, 6)]                # тільки трансформація
positives = [x for x in [-3, 5, -1, 8, 0, -7] if x > 0]  # тільки фільтр

print(squares)
print(positives)

# Правило: comprehension будує НОВИЙ список. Якщо мета — виконати дію (print,
# зберегти у файл) без побудови списку — використовуйте звичайний for.

[1, 4, 9, 16, 25]
[5, 8]


### 9. Dict comprehension — та сама ідея для словників

Знову спочатку звичайним циклом: словник `{слово: довжина_слова}`:

In [11]:
words = ["apple", "banana", "cherry", "date"]

word_lengths = {}
for w in words:
    word_lengths[w] = len(w)

print(word_lengths)
# {'apple': 5, 'banana': 6, 'cherry': 6, 'date': 4}

{'apple': 5, 'banana': 6, 'cherry': 6, 'date': 4}


In [12]:
word_lengths_v2 = {w: len(w) for w in words}

print(word_lengths_v2)
assert word_lengths_v2 == word_lengths
print("Той самий результат, що й циклом вище")

{'apple': 5, 'banana': 6, 'cherry': 6, 'date': 4}
Той самий результат, що й циклом вище


### Підсумок CONCEPT

| Патерн | Питання | Інструмент |
|---|---|---|
| Iteration | Як пройтись по всіх елементах? | `for` / `while` |
| Mapping | Як перетворити кожен елемент? | comprehension |
| Counting | Скільки разів зустрівся кожен варіант? | `d.get(key, 0) + 1` |
| Grouping | Які саме записи в кожній групі? | `d.setdefault(key, []).append(...)` |

Далі — застосовуємо саме ці інструменти (`dict`, `set`, `while`, `for`, nested comprehension), крок за кроком будуючи реальну гру.

## 🛠️ CREATE — будуємо Minesweeper крок за кроком

Правила гри: дошка 8×8, під випадковими клітинками сховано 10 бомб. Гравець вводить координати `рядок стовпець`. Якщо клітинка безпечна — на ній з'являється число (скільки бомб торкається її з 8 сусідніх + вона сама). Якщо клітинка — бомба, гра закінчується. Відкрити всі безпечні клітинки — перемога.

Будуємо весь скрипт **без жодної функції** — функції (`def`) будуть на Уроці 7, коли розкладемо цей самий скрипт на іменовані частини. Зараз кожен новий шматок логіки просто додається до попереднього.

### Крок 1 — позиції бомб

`bombs` — множина (`set`) кортежів `(row, col)`: множина сама відкидає дублікати, тож не потрібно вручну перевіряти, чи ця позиція вже зайнята. `while` — бо наперед невідомо, скільки випадкових спроб знадобиться, щоб набрати 10 **унікальних** позицій (та сама ідея, що в RETRIEVE вище).

In [13]:
import random

SIZE = 8
BOMBS = 10

random.seed(42)   # фіксуємо генератор — щоб приклад у конспекті завжди давав той самий результат

bombs = set()
while len(bombs) < BOMBS:
    bombs.add((random.randint(0, SIZE - 1), random.randint(0, SIZE - 1)))

print(f"Згенеровано {len(bombs)} бомб:")
print(bombs)

Згенеровано 10 бомб:
{(0, 1), (7, 4), (4, 3), (1, 1), (0, 3), (3, 3), (6, 0), (1, 0), (3, 2), (6, 3)}


### Крок 2 — прихована дошка

Дошка — список списків, усі клітинки спочатку `"."`. Спочатку звичайним подвійним циклом:

In [14]:
hidden_v1 = []
for row in range(SIZE):
    board_row = []
    for col in range(SIZE):
        board_row.append(".")
    hidden_v1.append(board_row)

print(hidden_v1[0])
print(f"Рядків: {len(hidden_v1)}, довжина кожного: {len(hidden_v1[0])}")

['.', '.', '.', '.', '.', '.', '.', '.']
Рядків: 8, довжина кожного: 8


⚠️ **Пастка, якої тут немає, але яка трапляється часто:** `[["."] * SIZE] * SIZE` виглядає як коротший запис того самого, але це **одна й та сама** внутрішня list-клітинка, повторена 8 разів — зміна однієї клітинки зіпсує всі рядки одразу. Подвійний цикл (і nested-comprehension нижче) щоразу створює **новий** список для кожного рядка — це і є правильна поведінка.

Та сама побудова — nested list comprehension (comprehension всередині comprehension), компактніше. Саме цю форму використовуємо далі:

In [15]:
hidden = [["." for col in range(SIZE)] for row in range(SIZE)]

print(hidden[0])
assert hidden == hidden_v1
print("Той самий результат, що й подвійним циклом вище")

['.', '.', '.', '.', '.', '.', '.', '.']
Той самий результат, що й подвійним циклом вище


### Крок 3 — друк дошки

Верхній рядок — номери колонок. Кожен наступний рядок — номер рядка + вміст клітинок:

In [16]:
header = "  "
for col in range(len(hidden)):
    header += " " + str(col)
print(header)

for row in range(len(hidden)):
    line = str(row) + " "
    for cell in hidden[row]:
        line += " " + cell
    print(line)

   0 1 2 3 4 5 6 7
0  . . . . . . . .
1  . . . . . . . .
2  . . . . . . . .
3  . . . . . . . .
4  . . . . . . . .
5  . . . . . . . .
6  . . . . . . . .
7  . . . . . . . .


### Крок 4 — зчитування ходу гравця

Читаємо `"рядок стовпець"`, розбиваємо на дві частини і перевіряємо: (1) що це взагалі два числа, (2) що вони в межах дошки. `while True` + `break`, коли ввід нарешті коректний. Перевіримо фрагмент окремо, підмінивши `input()` заздалегідь підготовленими відповідями (без реальної клавіатури) — три спроби: некоректний формат, вихід за межі дошки, і нарешті правильний хід:

In [17]:
import builtins

_fake_answers = iter(["abc x", "9 9", "1 1"])
_real_input = builtins.input
builtins.input = lambda prompt="": next(_fake_answers)

while True:
    answer = input("Row and column, for example 3 5: ").split()
    if len(answer) != 2 or not answer[0].isdigit() or not answer[1].isdigit():
        print("Type two numbers from 0 to", SIZE - 1)
    elif int(answer[0]) >= SIZE or int(answer[1]) >= SIZE:
        print("This cell is outside the board")
    else:
        row, col = int(answer[0]), int(answer[1])
        break

builtins.input = _real_input   # повертаємо звичайний input()

print(f"Прийнятий хід: row={row}, col={col}")
assert (row, col) == (1, 1)
print("OK — некоректний формат і вихід за межі дошки коректно пропущені")

Type two numbers from 0 to 7
This cell is outside the board
Прийнятий хід: row=1, col=1
OK — некоректний формат і вихід за межі дошки коректно пропущені


### Крок 5 — скільки бомб навколо клітинки

Перевіряємо всі 9 клітинок квадрата 3×3 навколо `(row, col)` (включно з самою клітинкою) — подвійний цикл по зсувах `-1, 0, 1`:

In [18]:
test_bombs = {(1, 1), (1, 2), (3, 3)}

for test_row, test_col in [(1, 1), (0, 0), (5, 5), (2, 2)]:
    around = 0
    for dr in (-1, 0, 1):
        for dc in (-1, 0, 1):
            if (test_row + dr, test_col + dc) in test_bombs:
                around += 1
    print(f"навколо ({test_row},{test_col}): {around}")

# (1,1) сама є бомбою + сусідня (1,2) -> 2
# (2,2) бачить (1,1), (1,2), (3,3) -> 3

навколо (1,1): 2
навколо (0,0): 1
навколо (5,5): 0
навколо (2,2): 3


### Крок 6 — одна ітерація ходу: три можливі результати

Тепер з'єднуємо кроки 1, 2, 4 і 5 в один цілісний шматок логіки, у правильному порядку: спочатку «чи вже відкрито», потім «чи це бомба», і лише тоді — підрахунок бомб навколо. Спробуємо на ході `(1, 1)`, прийнятому на Кроці 4:

In [19]:
if hidden[row][col] != ".":
    print("Ця клітинка вже відкрита")
elif (row, col) in bombs:
    print("Boom! Game over.")
else:
    around = 0
    for dr in (-1, 0, 1):
        for dc in (-1, 0, 1):
            if (row + dr, col + dc) in bombs:
                around += 1
    hidden[row][col] = str(around)
    print(f"Клітинка ({row}, {col}) відкрита: {around} бомб(и) навколо")

Boom! Game over.


Виявляється, хід `(1, 1)` з Кроку 4 влучив би прямо в бомбу — Крок 4 перевіряв лише формат і межі дошки, а не сам факт бомби, тому перевірка `(row, col) in bombs` — окремий, третій крок. Спробуємо тепер безпечну клітинку `(5, 5)`, якої немає серед `bombs`:

In [20]:
row, col = 5, 5
assert (row, col) not in bombs

if hidden[row][col] != ".":
    print("Ця клітинка вже відкрита")
elif (row, col) in bombs:
    print("Boom! Game over.")
else:
    around = 0
    for dr in (-1, 0, 1):
        for dc in (-1, 0, 1):
            if (row + dr, col + dc) in bombs:
                around += 1
    hidden[row][col] = str(around)
    print(f"Клітинка ({row}, {col}) відкрита: {around} бомб(и) навколо")

print(hidden[row])

Клітинка (5, 5) відкрита: 0 бомб(и) навколо
['.', '.', '.', '.', '.', '0', '.', '.']


Той самий код, викликаний ще раз на тій самій клітинці — тепер вона вже не `"."`, тож спрацьовує перша гілка:

In [21]:
if hidden[row][col] != ".":
    print("Ця клітинка вже відкрита")
elif (row, col) in bombs:
    print("Boom! Game over.")
else:
    around = 0
    for dr in (-1, 0, 1):
        for dc in (-1, 0, 1):
            if (row + dr, col + dc) in bombs:
                around += 1
    hidden[row][col] = str(around)
    print(f"Клітинка ({row}, {col}) відкрита: {around} бомб(и) навколо")

Ця клітинка вже відкрита


### Крок 7 — `while ... else` і повний ігровий цикл

Гра — це не один хід, а `while`, що триває, поки не відкрито всі безпечні клітинки. У RETRIEVE вище вже був `while` без `else`; тут `else` — нова частина: він виконується, тільки якщо `while` завершився природно (умова стала хибною), і НЕ виконується, якщо цикл перервано через `break`. Маленька ізольована демонстрація, ще без Minesweeper:

In [22]:
print("Без break — else виконується:")
n = 0
while n < 3:
    print("  n =", n)
    n += 1
else:
    print("  -> else: цикл завершився природно")

print()
print("З break — else НЕ виконується:")
n = 0
while n < 3:
    if n == 1:
        break
    print("  n =", n)
    n += 1
else:
    print("  -> else: цього рядка не буде")
print("  -> цикл перервано через break, без else")

Без break — else виконується:
  n = 0
  n = 1
  n = 2
  -> else: цикл завершився природно

З break — else НЕ виконується:
  n = 0
  -> цикл перервано через break, без else


Саме так `while ... else` працює в Minesweeper: цикл триває, поки `opened < SIZE * SIZE - BOMBS`; `break` (влучання в бомбу) пропускає `else`; природне завершення (усі безпечні клітинки відкрито) — виконує `else: print("You win!")`.

Тепер складаємо Кроки 1–6 (генерація бомб, дошка, друк, зчитування ходу, три гілки одного ходу) і `opened`/`while ... else` в один цільний скрипт. `input()` не читає реальну клавіатуру в ноутбуці, тож так само, як на Кроці 4, підміняємо його заготовленим списком ходів — тільки тепер на весь цикл, а не на одну спробу:

In [23]:
import io
import contextlib
import builtins


def play_with_moves(source, moves, seed):
    """Виконує сирий текст скрипту source, підмінивши input() списком moves.
    Повертає весь друкований вивід одним рядком."""
    it = iter(moves)
    real_input = builtins.input
    builtins.input = lambda prompt="": next(it)

    buffer = io.StringIO()
    random.seed(seed)
    try:
        with contextlib.redirect_stdout(buffer):
            exec(compile(source, "<student_game_source>", "exec"), {})
    finally:
        builtins.input = real_input

    return buffer.getvalue()


student_game_source = """
import random

SIZE = 8
BOMBS = 10

bombs = set()
while len(bombs) < BOMBS:
    bombs.add((random.randint(0, SIZE - 1), random.randint(0, SIZE - 1)))

hidden = [["." for col in range(SIZE)] for row in range(SIZE)]

opened = 0

while opened < SIZE * SIZE - BOMBS:
    header = "  "
    for col in range(len(hidden)):
        header += " " + str(col)
    print(header)
    for row in range(len(hidden)):
        line = str(row) + " "
        for cell in hidden[row]:
            line += " " + cell
        print(line)

    while True:
        answer = input("Row and column, for example 3 5: ").split()
        if len(answer) != 2 or not answer[0].isdigit() or not answer[1].isdigit():
            print("Type two numbers from 0 to", SIZE - 1)
        elif int(answer[0]) >= SIZE or int(answer[1]) >= SIZE:
            print("This cell is outside the board")
        else:
            row, col = int(answer[0]), int(answer[1])
            break

    if hidden[row][col] != ".":
        print("This cell is already open")
        continue
    if (row, col) in bombs:
        print("Boom! Game over.")
        break

    around = 0
    for dr in (-1, 0, 1):
        for dc in (-1, 0, 1):
            if (row + dr, col + dc) in bombs:
                around += 1

    hidden[row][col] = str(around)
    opened += 1
else:
    print("You win!")
"""

demo_moves = ["0 0", "0 0", "9 9", "abc x", "1 1", "2 2", "3 3"]
demo_output = play_with_moves(student_game_source, demo_moves, seed=42)
print(demo_output)

   0 1 2 3 4 5 6 7
0  . . . . . . . .
1  . . . . . . . .
2  . . . . . . . .
3  . . . . . . . .
4  . . . . . . . .
5  . . . . . . . .
6  . . . . . . . .
7  . . . . . . . .
   0 1 2 3 4 5 6 7
0  3 . . . . . . .
1  . . . . . . . .
2  . . . . . . . .
3  . . . . . . . .
4  . . . . . . . .
5  . . . . . . . .
6  . . . . . . . .
7  . . . . . . . .
This cell is already open
   0 1 2 3 4 5 6 7
0  3 . . . . . . .
1  . . . . . . . .
2  . . . . . . . .
3  . . . . . . . .
4  . . . . . . . .
5  . . . . . . . .
6  . . . . . . . .
7  . . . . . . . .
This cell is outside the board
Type two numbers from 0 to 7
Boom! Game over.



### Крок 8 — показ усіх бомб наприкінці

Гра закінчилась ("Boom!" вище) — тепер потрібно показати, де стояли всі бомби. Проходимо по `bombs` і ставимо `"*"` на відповідні клітинки `hidden`, потім друкуємо дошку тим самим фрагментом друку, що й на Кроці 3. Застосуємо це напряму до реального стану гри з Кроку 6 (де `(5, 5)` вже відкрито):

In [24]:
for row, col in bombs:
    hidden[row][col] = "*"

header = "  "
for col in range(len(hidden)):
    header += " " + str(col)
print(header)
for row in range(len(hidden)):
    line = str(row) + " "
    for cell in hidden[row]:
        line += " " + cell
    print(line)

   0 1 2 3 4 5 6 7
0  . * . * . . . .
1  * * . . . . . .
2  . . . . . . . .
3  . . * * . . . .
4  . . . * . . . .
5  . . . . . 0 . .
6  * . . * . . . .
7  . . . . * . . .


Тепер маємо всі шматки: генерація бомб (1), дошка (2), друк (3), зчитування ходу (4), одна ітерація ходу — відкрити / вже відкрито / бомба (6), `opened` і `while ... else` (7), показ бомб (цей крок). Дописуємо показ бомб у кінець `student_game_source` з Кроку 7 — це той самий текст скрипту, доповнений останнім шматком:

In [25]:
student_game_source += """
for row, col in bombs:
    hidden[row][col] = "*"

header = "  "
for col in range(len(hidden)):
    header += " " + str(col)
print(header)
for row in range(len(hidden)):
    line = str(row) + " "
    for cell in hidden[row]:
        line += " " + cell
    print(line)
"""

demo_output = play_with_moves(student_game_source, demo_moves, seed=42)
print(demo_output)

   0 1 2 3 4 5 6 7
0  . . . . . . . .
1  . . . . . . . .
2  . . . . . . . .
3  . . . . . . . .
4  . . . . . . . .
5  . . . . . . . .
6  . . . . . . . .
7  . . . . . . . .
   0 1 2 3 4 5 6 7
0  3 . . . . . . .
1  . . . . . . . .
2  . . . . . . . .
3  . . . . . . . .
4  . . . . . . . .
5  . . . . . . . .
6  . . . . . . . .
7  . . . . . . . .
This cell is already open
   0 1 2 3 4 5 6 7
0  3 . . . . . . .
1  . . . . . . . .
2  . . . . . . . .
3  . . . . . . . .
4  . . . . . . . .
5  . . . . . . . .
6  . . . . . . . .
7  . . . . . . . .
This cell is outside the board
Type two numbers from 0 to 7
Boom! Game over.
   0 1 2 3 4 5 6 7
0  3 * . * . . . .
1  * * . . . . . .
2  . . . . . . . .
3  . . * * . . . .
4  . . . * . . . .
5  . . . . . . . .
6  * . . * . . . .
7  . . . . * . . .



### Крок 9 — звірка з файлом Уроку 7

Ми щойно самостійно написали повний скрипт крок за кроком. Лишилось перевірити: чи це справді той самий скрипт, яким скористається Урок 7? Відкриваємо `minesweeper_before_refactor.py` (сусідній урок) і порівнюємо вивід обох версій під однаковим `seed` і однаковим списком ходів — не "на око", а побайтовим `assert`:

In [26]:
with open(
    "../lesson_07_functions/resources/minesweeper_before_refactor.py",
    encoding="utf-8",
) as f:
    full_game_source = f.read()

output_file = play_with_moves(full_game_source, demo_moves, seed=42)
output_student = play_with_moves(student_game_source, demo_moves, seed=42)

assert output_file == output_student, "Наш скрипт поводиться інакше, ніж файл Уроку 7!"
print("✅ Побайтово однаковий вивід — ми справді побудували той самий скрипт, який відкриває Урок 7.")

✅ Побайтово однаковий вивід — ми справді побудували той самий скрипт, який відкриває Урок 7.


### ⚙️ Перевірка поведінки гри (не частина гри — інструмент для контролю якості конспекту)

Клітинки нижче не вчать нічого нового — вони автоматично підтверджують, що гра справді програється й вигравається правильно, перш ніж рухатись до Уроку 7.

In [27]:
# Сценарій "програш": підбираємо ходи так, щоб перший же хід влучив у бомбу
random.seed(7)
probe_bombs = set()
while len(probe_bombs) < BOMBS:
    probe_bombs.add((random.randint(0, SIZE - 1), random.randint(0, SIZE - 1)))
bomb_row, bomb_col = next(iter(probe_bombs))

lose_output = play_with_moves(full_game_source, [f"{bomb_row} {bomb_col}"], seed=7)
print(lose_output)

assert "Boom! Game over." in lose_output
print("OK — сценарій програшу підтверджено")

   0 1 2 3 4 5 6 7
0  . . . . . . . .
1  . . . . . . . .
2  . . . . . . . .
3  . . . . . . . .
4  . . . . . . . .
5  . . . . . . . .
6  . . . . . . . .
7  . . . . . . . .
Boom! Game over.
   0 1 2 3 4 5 6 7
0  . . . . . . * .
1  . * . * . . * .
2  . . . . . . . .
3  * * . . . . . .
4  . . . . . . . .
5  * . * . . . . .
6  * * . . . . . .
7  . . . . . . . .

OK — сценарій програшу підтверджено


In [28]:
# Сценарій "перемога": перебираємо ВСІ безпечні клітинки для фіксованого seed
random.seed(99)
win_bombs = set()
while len(win_bombs) < BOMBS:
    win_bombs.add((random.randint(0, SIZE - 1), random.randint(0, SIZE - 1)))

win_moves = [
    f"{r} {c}"
    for r in range(SIZE)
    for c in range(SIZE)
    if (r, c) not in win_bombs
]

win_output = play_with_moves(full_game_source, win_moves, seed=99)

assert "You win!" in win_output
assert "Boom! Game over." not in win_output
print(f"OK — повна перемога за {len(win_moves)} ходів підтверджена")

OK — повна перемога за 54 ходів підтверджена


### Звірка з Уроком 7

Урок 7 рефакторить файл `minesweeper_before_refactor.py` у функції і звіряє вивід «до» і «після» побайтово (той самий `seed=42` і той самий список ходів, що ми використовували на Кроках 7–9). На Кроці 9 ми довели те саме про цей урок: скрипт, який ми щойно написали крок за кроком (Кроки 1–8), і реальний файл, яким скористається Урок 7, дають побайтово однаковий вивід — тобто ми справді самостійно побудували ту саму гру, а не просто подивились на готовий файл.

## 🔄 TRANSFER — самостійне розширення: прапорці

Додай гравцю можливість **позначити клітинку прапорцем** замість того, щоб одразу її відкривати — так гравець запам'ятовує підозрілі клітинки, не ризикуючи підірватись.

Вимоги:
- Хід `"f 3 5"` (буква `f`, потім координати) — ставить прапорець на `(3, 5)`, **не відкриваючи** клітинку.
- Прапорці зберігай у `dict` `flags` — ключ `(row, col)`, значення не важливе (`True`).
- На дошці клітинка з прапорцем показує `"F"` замість `"."`.
- Хід без `"f"` (звичайні координати) на клітинці з прапорцем — знімає прапорець замість відкриття (не втрачаючи хід).
- Після кожного ходу друкуй, скільки прапорців зараз виставлено — **через comprehension**, а не окремий лічильник: `sum(1 for _ in flags)` або просто `len(flags)`.

Це окремий варіант скрипту — `minesweeper_before_refactor.py` не змінюється.

In [29]:
FLAG_GAME_TEMPLATE = '''
import random

SIZE = 8
BOMBS = 10

random.seed(SEED_PLACEHOLDER)

bombs = set()
while len(bombs) < BOMBS:
    bombs.add((random.randint(0, SIZE - 1), random.randint(0, SIZE - 1)))

hidden = [["." for col in range(SIZE)] for row in range(SIZE)]
flags = {}
opened = 0

while opened < SIZE * SIZE - BOMBS:
    header = "  "
    for col in range(len(hidden)):
        header += " " + str(col)
    print(header)
    for row in range(len(hidden)):
        line = str(row) + " "
        for c_idx, cell in enumerate(hidden[row]):
            symbol = "F" if (row, c_idx) in flags and cell == "." else cell
            line += " " + symbol
        print(line)
    print(f"Прапорців виставлено: {len(flags)}")

    raw = input("Row and column, or f ROW COL: ").split()

    # YOUR CODE HERE
    # BEGIN SOLUTION
    if len(raw) == 3 and raw[0] == "f" and raw[1].isdigit() and raw[2].isdigit():
        row, col = int(raw[1]), int(raw[2])
        if 0 <= row < SIZE and 0 <= col < SIZE:
            if (row, col) in flags:
                del flags[(row, col)]
            else:
                flags[(row, col)] = True
        continue

    if len(raw) != 2 or not raw[0].isdigit() or not raw[1].isdigit():
        print("Type two numbers from 0 to", SIZE - 1)
        continue
    row, col = int(raw[0]), int(raw[1])
    if row >= SIZE or col >= SIZE:
        print("This cell is outside the board")
        continue
    if (row, col) in flags:
        del flags[(row, col)]
        continue
    # END SOLUTION

    if hidden[row][col] != ".":
        print("This cell is already open")
        continue
    if (row, col) in bombs:
        print("Boom! Game over.")
        break

    around = 0
    for dr in (-1, 0, 1):
        for dc in (-1, 0, 1):
            if (row + dr, col + dc) in bombs:
                around += 1
    hidden[row][col] = str(around)
    opened += 1
else:
    print("You win!")

for row, col in bombs:
    hidden[row][col] = "*"
header = "  "
for col in range(len(hidden)):
    header += " " + str(col)
print(header)
for row in range(len(hidden)):
    line = str(row) + " "
    for cell in hidden[row]:
        line += " " + cell
    print(line)
'''

flag_source = FLAG_GAME_TEMPLATE.replace("SEED_PLACEHOLDER", "42")

flag_moves = ["f 0 0", "1 1", "0 0", "2 2"]
flag_output = play_with_moves(flag_source, flag_moves, seed=42)
print(flag_output)

assert "Прапорців виставлено: 1" in flag_output
print("OK — прапорець виставлено, знято і скасовано коректно")

   0 1 2 3 4 5 6 7
0  . . . . . . . .
1  . . . . . . . .
2  . . . . . . . .
3  . . . . . . . .
4  . . . . . . . .
5  . . . . . . . .
6  . . . . . . . .
7  . . . . . . . .
Прапорців виставлено: 0
   0 1 2 3 4 5 6 7
0  F . . . . . . .
1  . . . . . . . .
2  . . . . . . . .
3  . . . . . . . .
4  . . . . . . . .
5  . . . . . . . .
6  . . . . . . . .
7  . . . . . . . .
Прапорців виставлено: 1
Boom! Game over.
   0 1 2 3 4 5 6 7
0  . * . * . . . .
1  * * . . . . . .
2  . . . . . . . .
3  . . * * . . . .
4  . . . * . . . .
5  . . . . . . . .
6  * . . * . . . .
7  . . . . * . . .

OK — прапорець виставлено, знято і скасовано коректно


## ✅ Самоперевірка (5 запитань)

**1.** `for k, v in scores:` (без `.items()`) падає з `ValueError`. Чому саме `ValueError`, а не, наприклад, `TypeError`?

<details><summary>Відповідь</summary><code>for key in d</code> дає рядки-ключі; Python намагається розпакувати кожен такий рядок у дві змінні <code>k, v</code> — це вдається лише для рядків рівно з 2 символів, інакше кількість елементів для розпакування не збігається з кількістю змінних, а це саме <code>ValueError</code> ("too many/not enough values to unpack").</details>

**2.** `d.setdefault('x', []).append(5)` — що станеться, якщо викликати цей самий рядок ще раз одразу після?

<details><summary>Відповідь</summary>Ключ <code>'x'</code> вже є, тож <code>setdefault</code> нічого не створює заново — просто повертає існуючий список, і <code>.append(5)</code> додає до нього другий елемент: <code>d == {'x': [5, 5]}</code>.</details>

**3.** Напиши list comprehension, який з `range(1, 21)` залишає лише числа, кратні 3.

<details><summary>Відповідь</summary><code>[x for x in range(1, 21) if x % 3 == 0]</code></details>

**4.** `hidden = [["."] * SIZE] * SIZE` — у чому конкретно небезпека цього запису?

<details><summary>Відповідь</summary>Зовнішній список повторює **один і той самий** внутрішній список-об'єкт 8 разів (не 8 незалежних копій) — зміна однієї клітинки (<code>hidden[0][0] = "1"</code>) миттєво "зіпсує" той самий індекс у всіх інших рядках, бо це фізично один об'єкт у пам'яті.</details>

**5.** Чому в Minesweeper-скрипті цього уроку немає жодної функції (`def`)?

<details><summary>Відповідь</summary>Функції — тема наступного уроку. Тут навмисно весь код лежить на одному рівні модуля, щоб на Уроці 7 було видно і відчутно, навіщо декомпозиція взагалі потрібна (спільний простір імен, повторення коду друку дошки двічі тощо) — рефакторинг має сенс тільки тоді, коли є що рефакторити.</details>

### Шпаргалка

```python
# dict
d[key]                    # KeyError, якщо немає
d.get(key, default)       # безпечне читання
d.setdefault(key, default)  # читання + створення, якщо немає
for k in d: ...            # тільки ключі
for k, v in d.items(): ... # пари

# Counting / Grouping
d[key] = d.get(key, 0) + 1          # підрахунок
d.setdefault(key, []).append(item)  # групування

# Comprehensions
[expr for x in seq if cond]   # list
{k: v for k, v in seq}        # dict
{expr for x in seq}           # set
```

## 🍽️ Другий наскрізний приклад: аналітика ресторану

У попередньому уроці (позиція 5 — списки, кортежі, множини) ми перетворили 244 реальних чеки ресторану на `orders: List[Order]`, де `Order` — це `NamedTuple` з полями `total_bill, tip, sex, smoker, day, time, size`. Тут ми продовжуємо саме з цієї точки — агрегуємо ці чеки словниками, шукаємо лідера, групуємо і фільтруємо через comprehensions. Це та сама модель мислення, що й у Minesweeper вище (дані → накопичення в dict → результат), але на іншому домені — бізнес-аналітика замість гри.

Коротко повторюємо з попереднього уроку, щоб продовжити тут:

In [ ]:
import seaborn as sns
from typing import NamedTuple


class Order(NamedTuple):
    total_bill: float
    tip: float
    sex: str
    smoker: str
    day: str
    time: str
    size: int


tips_df = sns.load_dataset("tips")

orders = [
    Order(
        total_bill=row["total_bill"],
        tip=row["tip"],
        sex=row["sex"],
        smoker=row["smoker"],
        day=row["day"],
        time=row["time"],
        size=row["size"],
    )
    for _, row in tips_df.iterrows()
]

print(f"Завантажено: {len(orders)} чеків")
print(orders[0])

### `for` як обробник потоку подій

Кожна ітерація `for order in orders` — це один клієнт, що заплатив рахунок. Спочатку простий прохід (дивимось на дані), потім накопичувач (Aggregation pattern) — той самий патерн, що й `while len(bombs) < BOMBS` у Minesweeper, тільки тепер накопичуємо суму, а не кількість:

In [ ]:
print("Перші 5 чеків:")
for order in orders[:5]:
    tip_pct = (order.tip / order.total_bill) * 100
    print(f"{order.day:6} | {order.time:7} | ${order.total_bill:6.2f} | tip {tip_pct:.1f}%")

print()

total_revenue = 0
total_tips = 0
total_guests = 0

for order in orders:
    total_revenue += order.total_bill
    total_tips += order.tip
    total_guests += order.size

avg_bill = total_revenue / len(orders)

print(f"Загальний дохід:  ${total_revenue:.2f}")
print(f"Загальні чайові:  ${total_tips:.2f}")
print(f"Всього гостей:    {total_guests}")
print(f"Середній чек:     ${avg_bill:.2f}")

### Counting pattern: `d[key] = d.get(key, 0) + value`

Найпоширеніший алгоритм бізнес-аналітики. Спочатку побачимо, чому наївний варіант падає:

In [ ]:
# Наївний варіант — падає, бо ключа "apple" ще немає
fruits = {}
try:
    fruits["apple"] = fruits["apple"] + 1
except KeyError as e:
    print(f"KeyError: {e} — ключа ще немає, нема що додавати")

# Виправлено: .get(key, 0) підставляє 0, якщо ключа немає
fruits = {}
fruits["apple"] = fruits.get("apple", 0) + 1
print(fruits)

Живий перегляд — як словник оновлюється на перших 10 чеках, крок за кроком:

In [ ]:
revenue_by_day = {}

print("Крок | День  | Чек     | Стан dict")
for i, order in enumerate(orders[:10], 1):
    day = order.day
    bill = order.total_bill
    revenue_by_day[day] = revenue_by_day.get(day, 0) + bill
    state = {k: f"${v:.0f}" for k, v in revenue_by_day.items()}
    print(f"  {i:2} | {day:5} | ${bill:6.2f} | {state}")

Тепер по всіх 244 чеках — три словники за один цикл (revenue, кількість чеків, гостей по днях):

In [ ]:
revenue_by_day = {}
orders_by_day = {}
guests_by_day = {}

for order in orders:
    day = order.day
    revenue_by_day[day] = revenue_by_day.get(day, 0) + order.total_bill
    orders_by_day[day] = orders_by_day.get(day, 0) + 1
    guests_by_day[day] = guests_by_day.get(day, 0) + order.size

print("День  | Дохід     | Чеків | Гостей")
for day in revenue_by_day:
    print(f"{day:6} | ${revenue_by_day[day]:8.2f} | {orders_by_day[day]:5} | {guests_by_day[day]:6}")

assert sum(orders_by_day.values()) == len(orders)
assert abs(sum(revenue_by_day.values()) - total_revenue) < 0.01
print("\nOK — сума по днях збігається із загальним доходом і кількістю чеків")

### Leader algorithm: який день найприбутковіший?

Ручний алгоритм «поточний чемпіон», потім пітонічний варіант через `max(d, key=d.get)` — і перевіряємо, що обидва дають той самий результат:

In [ ]:
# Варіант 1 — ручний алгоритм «поточний чемпіон»
best_day_manual = None
best_revenue_manual = 0
for day, revenue in revenue_by_day.items():
    if revenue > best_revenue_manual:
        best_revenue_manual = revenue
        best_day_manual = day

# Варіант 2 — пітонічний через max()
best_day = max(revenue_by_day, key=revenue_by_day.get)

assert best_day == best_day_manual
print(f"Найкращий день: {best_day} — ${revenue_by_day[best_day]:.2f}")

days_sorted = sorted(revenue_by_day.items(), key=lambda x: x[1], reverse=True)
print("\nРейтинг днів:")
for i, (day, rev) in enumerate(days_sorted, 1):
    bar = "#" * int(rev / 80)
    print(f"  {i}. {day:6}: ${rev:7.2f}  {bar}")

### Той самий Counting/Leader патерн — інший ключ

Той самий алгоритм, застосований до `order.time` замість `order.day`, без жодної зміни логіки — тільки поле, по якому групуємо:

In [ ]:
revenue_by_time = {}
orders_by_time = {}

for order in orders:
    t = order.time
    revenue_by_time[t] = revenue_by_time.get(t, 0) + order.total_bill
    orders_by_time[t] = orders_by_time.get(t, 0) + 1

print("Час     | Дохід      | Чеків | Avg чек")
for t in revenue_by_time:
    n = orders_by_time[t]
    rev = revenue_by_time[t]
    print(f"{t:7} | ${rev:9.2f} | {n:5} | ${rev/n:7.2f}")

winner = max(revenue_by_time, key=revenue_by_time.get)
assert abs(sum(revenue_by_time.values()) - total_revenue) < 0.01
print(f"\nПереможець: {winner}")

### `set` — унікальні значення без ручної перевірки

Ручний збір через `set.add()`, потім та сама ідея через set comprehension (з CONCEPT вище) — і перевіряємо рівність:

In [ ]:
unique_days_manual = set()
for order in orders:
    unique_days_manual.add(order.day)

unique_days = {order.day for order in orders}
unique_times = {order.time for order in orders}
unique_sizes = {order.size for order in orders}

assert unique_days == unique_days_manual
print("Дні роботи:    ", unique_days)
print("Типи зміни:    ", unique_times)
print("Розміри столів:", sorted(unique_sizes))

### Grouping pattern: `dict.setdefault(key, []).append(item)`

Різниця з Counting: не «скільки», а «які саме» — зберігаємо самі об'єкти `Order`, не лише лічильник:

In [ ]:
orders_by_day = {}
for order in orders:
    orders_by_day.setdefault(order.day, []).append(order)

print("День  | Чеків | Avg чек | Max чек")
for day, day_orders in orders_by_day.items():
    bills = [o.total_bill for o in day_orders]
    print(f"{day:6} | {len(day_orders):5} | ${sum(bills)/len(bills):6.2f}  | ${max(bills):.2f}")

assert sum(len(v) for v in orders_by_day.values()) == len(orders)
assert set(orders_by_day.keys()) == unique_days
print("\nOK — сума груп дорівнює кількості чеків, ключі збігаються з унікальними днями")

### `defaultdict` — той самий алгоритм, чистіший синтаксис

`defaultdict(int)`/`defaultdict(float)`/`defaultdict(list)` автоматично підставляють значення за замовчуванням для нового ключа — не треба ні `if`, ні `.get()`, ні `.setdefault()`. Переписуємо всі три патерни й перевіряємо, що результат ідентичний тому, що вже порахували вище:

In [ ]:
from collections import defaultdict

# Counting — раніше: orders_by_time[t] = orders_by_time.get(t, 0) + 1
orders_by_time_dd = defaultdict(int)
for order in orders:
    orders_by_time_dd[order.time] += 1

# Aggregation — раніше: revenue_by_day[day] = revenue_by_day.get(day, 0) + bill
revenue_by_day_dd = defaultdict(float)
for order in orders:
    revenue_by_day_dd[order.day] += order.total_bill

# Grouping — раніше: orders_by_day.setdefault(day, []).append(order)
groups = defaultdict(list)
for order in orders:
    groups[order.day].append(order)

assert dict(orders_by_time_dd) == orders_by_time
assert {k: round(v, 2) for k, v in revenue_by_day_dd.items()} == {k: round(v, 2) for k, v in revenue_by_day.items()}
assert {k: len(v) for k, v in groups.items()} == {k: len(v) for k, v in orders_by_day.items()}
print("OK — defaultdict дає той самий результат, що й .get()/.setdefault() варіанти")

### `Counter` — третій спосіб рахувати

Коли єдине, що потрібно — це «скільки разів щось зустрілось», `collections.Counter` ще компактніший за `defaultdict(int)`: приймає готову послідовність і одразу рахує, плюс дає `.most_common()`:

In [ ]:
from collections import Counter

orders_count_counter = Counter(order.day for order in orders)

print("Кількість чеків по днях (найпопулярніші перші):")
for day, count in orders_count_counter.most_common():
    print(f"  {day:6}: {count} чеки")

# Порівнюємо з тим самим підрахунком через .get() — мають збігтись
orders_count_get = {}
for order in orders:
    orders_count_get[order.day] = orders_count_get.get(order.day, 0) + 1

assert dict(orders_count_counter) == orders_count_get
print("\nOK — Counter і .get() дають однаковий підрахунок по днях")

### Comprehensions на реальних даних: фільтр, трансформація, топ-N

Застосовуємо list comprehensions (з CONCEPT вище) до `orders` — фільтр, мапінг, і фільтр+трансформація разом:

In [ ]:
# Filter: тільки умова
big_orders = [o for o in orders if o.total_bill > 40]
print(f"Чеків понад $40: {len(big_orders)}")

# Mapping: трансформація (tip% для кожного чеку)
all_tip_percents = [(o.tip / o.total_bill) * 100 for o in orders]
avg_tip = sum(all_tip_percents) / len(all_tip_percents)
print(f"Tip %: avg={avg_tip:.1f}%  min={min(all_tip_percents):.1f}%  max={max(all_tip_percents):.1f}%")

# Filter + Transform
dinner_bills = [o.total_bill for o in orders if o.time == "Dinner"]
print(f"Dinner чеків: {len(dinner_bills)}, avg ${sum(dinner_bills)/len(dinner_bills):.2f}")

# Топ-10 найбільших чеків — sorted() + slice
top10 = sorted(orders, key=lambda o: o.total_bill, reverse=True)[:10]
print("\nТоп-3 з топ-10:")
for o in top10[:3]:
    print(f"  {o.day:6} {o.time:7} ${o.total_bill:6.2f}")

assert len(big_orders) + len([o for o in orders if o.total_bill <= 40]) == len(orders)
assert len(dinner_bills) == orders_by_time["Dinner"]
print("\nOK — фільтр + доповнення до нього дають повну множину чеків")

### Dict comprehension: середній чек і середні чайові по днях

Спочатку звичайний цикл (для перевірки), потім dict comprehension — і порівнюємо, що результат той самий:

In [ ]:
# Кількість чеків по днях (Counting — уже знайомий патерн)
count_by_day = {}
for order in orders:
    count_by_day[order.day] = count_by_day.get(order.day, 0) + 1

# Звичайним циклом
avg_bill_by_day_loop = {}
for day in revenue_by_day:
    avg_bill_by_day_loop[day] = revenue_by_day[day] / count_by_day[day]

# Dict comprehension — та сама ідея, один вираз
avg_bill_by_day = {day: revenue_by_day[day] / count_by_day[day] for day in revenue_by_day}

assert avg_bill_by_day == avg_bill_by_day_loop
print("Середній чек по днях:")
for day, avg in avg_bill_by_day.items():
    print(f"  {day:6}: ${avg:.2f}")

### Dict comprehension: середні чайові (%) по днях

Використовуємо `groups` (`defaultdict(list)` з попереднього кроку) — для кожного дня беремо список чеків і рахуємо середній tip% спочатку циклом, потім comprehension:

In [ ]:
tip_pct_by_day_loop = {}
for day, day_orders in groups.items():
    tip_pct_by_day_loop[day] = sum((o.tip / o.total_bill) * 100 for o in day_orders) / len(day_orders)

tip_pct_by_day = {
    day: sum((o.tip / o.total_bill) * 100 for o in day_orders) / len(day_orders)
    for day, day_orders in groups.items()
}

assert tip_pct_by_day.keys() == tip_pct_by_day_loop.keys()
for day in tip_pct_by_day:
    assert abs(tip_pct_by_day[day] - tip_pct_by_day_loop[day]) < 1e-9

print("Середній % чайових по днях:")
for day, pct in sorted(tip_pct_by_day.items(), key=lambda x: x[1], reverse=True):
    print(f"  {day:6}: {pct:.2f}%")

### Фінальний звіт: зводимо все докупи

Один прохід по вже готових агрегатах (`revenue_by_day`, `total_revenue`, `total_tips`, `total_guests`, `avg_bill`) плюс два нові фільтри (обід/вечеря, розмір столу) — усе тим самим патерном, що й вище, зібране у формат «звіт для власника»:

In [ ]:
print("=" * 60)
print("        ЗВІТ ДЛЯ ВЛАСНИКА РЕСТОРАНУ")
print("=" * 60)

avg_tip_pct = sum((o.tip / o.total_bill) * 100 for o in orders) / len(orders)

print(f"\nЗАГАЛЬНА СТАТИСТИКА")
print(f"  Всього чеків:         {len(orders)}")
print(f"  Всього гостей:        {total_guests}")
print(f"  Загальний дохід:      ${total_revenue:.2f}")
print(f"  Загальні чайові:      ${total_tips:.2f}")
print(f"  Середній чек:         ${avg_bill:.2f}")
print(f"  Середній tip %:       {avg_tip_pct:.1f}%")

days_rank = sorted(revenue_by_day.items(), key=lambda x: x[1], reverse=True)
print(f"\nНАЙКРАЩІ ДНІ")
for i, (day, rev) in enumerate(days_rank, 1):
    marker = "  <- найкращий!" if i == 1 else ""
    print(f"  {i}. {day:6}: ${rev:.2f}{marker}")

rev_lunch = sum(o.total_bill for o in orders if o.time == "Lunch")
rev_dinner = sum(o.total_bill for o in orders if o.time == "Dinner")
cnt_lunch = sum(1 for o in orders if o.time == "Lunch")
cnt_dinner = sum(1 for o in orders if o.time == "Dinner")

print(f"\nОБІД vs ВЕЧЕРЯ")
print(f"  Обід:   {cnt_lunch:3} чеки | ${rev_lunch:.2f}  | avg ${rev_lunch/cnt_lunch:.2f}")
print(f"  Вечеря: {cnt_dinner:3} чеки | ${rev_dinner:.2f} | avg ${rev_dinner/cnt_dinner:.2f}")
print(f"  Переможець: {'Вечеря' if rev_dinner > rev_lunch else 'Обід'}")

avg_size = total_guests / len(orders)
large_cnt = sum(1 for o in orders if o.size >= 5)

print(f"\nСТОЛИ")
print(f"  Середній розмір: {avg_size:.1f} осіб")
print(f"  Великих (5+):    {large_cnt}")

top3 = sorted(orders, key=lambda o: o.total_bill, reverse=True)[:3]
print(f"\nТОП-3 ЧЕКИ")
for i, o in enumerate(top3, 1):
    tip_pct = (o.tip / o.total_bill) * 100
    print(f"  {i}. ${o.total_bill:.2f} | {o.day} {o.time} | {o.size} осіб | tip {tip_pct:.1f}%")

print("\n" + "=" * 60)

assert abs((rev_lunch + rev_dinner) - total_revenue) < 0.01
assert cnt_lunch + cnt_dinner == len(orders)
assert abs(sum(revenue_by_day.values()) - total_revenue) < 0.01
print("OK — обід+вечеря і сума по днях збігаються із загальним доходом")

Цей самий звіт легко перетворити на дашборд (`matplotlib`/`seaborn`: стовпчики по днях, обід vs вечеря, розподіл чеків) — тут це поза межами уроку, лише згадка.

## 🎯 TRANSFER: 5 самостійних задач на реальних даних

Той самий `orders`, тільки `for`/`dict`/`list`/`set`/comprehensions — без pandas. Розв'язки заховані під `# BEGIN SOLUTION` / `# END SOLUTION` — спробуй спочатку сам.

### Задача 1 — tip% по днях, відсортовано

Порахуй середній tip% для кожного дня (`{день: avg tip%}`), виведи відсортованим за спаданням.

In [ ]:
def tip_pct_report(orders: list) -> dict:
    """Повертає {день: середній tip%}."""
    # YOUR CODE HERE
    # BEGIN SOLUTION
    by_day = defaultdict(list)
    for o in orders:
        by_day[o.day].append(o)
    return {
        day: sum((o.tip / o.total_bill) * 100 for o in day_orders) / len(day_orders)
        for day, day_orders in by_day.items()
    }
    # END SOLUTION


result = tip_pct_report(orders)
print("День   | avg tip%")
for day, pct in sorted(result.items(), key=lambda x: x[1], reverse=True):
    print(f"{day:6} | {pct:.2f}%")

assert round(result["Fri"], 2) == 16.99
assert round(result["Sun"], 2) == 16.69
print("OK")

### Задача 2 — чоловіки vs жінки

Порівняй середній чек і середній tip% для `sex == "Male"` і `sex == "Female"`.

In [ ]:
def sex_report(orders: list, sex: str) -> tuple:
    """Повертає (кількість чеків, середній чек, середній tip%) для заданої статі."""
    # YOUR CODE HERE
    # BEGIN SOLUTION
    subset = [o for o in orders if o.sex == sex]
    avg_bill = sum(o.total_bill for o in subset) / len(subset)
    avg_tip_pct = sum((o.tip / o.total_bill) * 100 for o in subset) / len(subset)
    return len(subset), avg_bill, avg_tip_pct
    # END SOLUTION


for sex in ("Male", "Female"):
    count, avg_bill_sex, avg_tip_pct_sex = sex_report(orders, sex)
    print(f"{sex:6}: {count:3} чеків | avg ${avg_bill_sex:.2f} | avg tip {avg_tip_pct_sex:.2f}%")

male_count, male_avg_bill, _ = sex_report(orders, "Male")
female_count, female_avg_bill, _ = sex_report(orders, "Female")
assert male_count == 157
assert female_count == 87
assert round(male_avg_bill, 2) == 20.74
assert round(female_avg_bill, 2) == 18.06
print("OK")

### Задача 3 — аналіз по розміру столу

Для кожного `size` (1–6) — кількість чеків і середній чек **на людину** (`total_bill / size`).

In [ ]:
def size_report(orders: list) -> dict:
    """Повертає {size: (кількість чеків, середній чек на людину)}."""
    # YOUR CODE HERE
    # BEGIN SOLUTION
    by_size = defaultdict(list)
    for o in orders:
        by_size[o.size].append(o)
    return {
        size: (len(size_orders), sum(o.total_bill / o.size for o in size_orders) / len(size_orders))
        for size, size_orders in by_size.items()
    }
    # END SOLUTION


result = size_report(orders)
print("Розмір | Чеків | Avg на людину")
for size in sorted(result):
    count, avg_per_person = result[size]
    print(f"{size:6} | {count:5} | ${avg_per_person:.2f}")

assert result[2][0] == 156
assert round(result[1][1], 2) == 7.24
print("OK")

### Задача 4 — топ-5 найщедріших клієнтів

5 чеків із найвищим `tip%` (саме відсотком, не сумою чайових).

In [ ]:
def top_tippers(orders: list, n: int = 5) -> list:
    """Повертає n чеків із найвищим tip%, відсортованих за спаданням."""
    # YOUR CODE HERE
    # BEGIN SOLUTION
    return sorted(orders, key=lambda o: o.tip / o.total_bill, reverse=True)[:n]
    # END SOLUTION


top5 = top_tippers(orders)
print("День   | Час     | Сума   | tip%")
for o in top5:
    tip_pct = (o.tip / o.total_bill) * 100
    print(f"{o.day:6} | {o.time:7} | ${o.total_bill:6.2f} | {tip_pct:.1f}%")

assert len(top5) == 5
assert round((top5[0].tip / top5[0].total_bill) * 100, 1) == 71.0
print("OK")

### Задача 5* — матриця день × час

`{день: {"Lunch": дохід, "Dinner": дохід}}` — вкладений `defaultdict(lambda: defaultdict(float))`.

In [ ]:
def day_time_matrix(orders: list) -> dict:
    """Повертає {день: {час: дохід}}."""
    # YOUR CODE HERE
    # BEGIN SOLUTION
    matrix = defaultdict(lambda: defaultdict(float))
    for o in orders:
        matrix[o.day][o.time] += o.total_bill
    return matrix
    # END SOLUTION


matrix = day_time_matrix(orders)
print("День   | Lunch     | Dinner")
for day in matrix:
    lunch = matrix[day].get("Lunch", 0)
    dinner = matrix[day].get("Dinner", 0)
    print(f"{day:6} | ${lunch:8.2f} | ${dinner:8.2f}")

matrix_total = sum(v for day_totals in matrix.values() for v in day_totals.values())
assert abs(matrix_total - total_revenue) < 0.01
print("OK — сума матриці збігається із загальним доходом")

## Далі

Урок 7 («Функції») бере цей самий Minesweeper-скрипт (`resources/minesweeper_before_refactor.py` у сусідньому уроці) і розкладає його на п'ять функцій — крок за кроком, з перевіркою на кожному кроці, аж до побайтового збігу виводу «до» і «після» рефакторингу.